# Portfolio Project Part 2: Market Basket Analysis & Product Recommendations
This notebook uses Association Rule Mining (Apriori Algorithm) on transactional data to discover products frequently purchased together, providing actionable cross-selling and bundling recommendations.

In [1]:
# Import core data processing and visualization tools
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Install and import mlxtend for Association Rule Mining
!pip install mlxtend --quiet
from mlxtend.frequent_patterns import apriori, association_rules

print("Setup completed successfully.")

Setup completed successfully.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
import os
import pandas as pd
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define path and load CSVs
base_path = '/content/drive/My Drive/e_commerce_database/'

orders = pd.read_csv(os.path.join(base_path, 'orders.csv'))
products = pd.read_csv(os.path.join(base_path, 'products.csv'))

# 3. Merge order items with product details
df_basket = orders.merge(products, on='ProductID', how='left')
df_basket.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,OrderID,CustomerID,OrderDate,ProductID,Quantity,Discount,PaymentMethod,Status,ProductName,Category,UnitPrice
0,500001,103695,2025-08-28,2003,4.0,0.1,Gateway,Completed,USB-C Cable,Accessories,9.0
1,500002,107935,2024-05-31,2014,1.0,0.1,Wallet,Completed,Backpack,Accessories,42.0
2,500003,105141,2025-08-10,2002,1.0,0.2,CardToCard,Completed,Mechanical Keyboard,Electronics,62.0
3,500004,108780,2024-10-11,2012,1.0,0.1,Gateway,Completed,Office Chair,Home Office,180.0
4,500005,101187,2024-02-19,2008,2.0,0.0,Gateway,Completed,Phone Case,Accessories,14.0


# Data Preprocessing: Reshaping to Transaction Basket Format
Transforming transaction rows into a 1/0 basket matrix where each row represents an order and each column represents whether a product was included.

In [3]:
# 1. Pivot orders into a One-Hot Encoded transaction matrix
basket = (df_basket.groupby(['OrderID', 'ProductID'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('OrderID'))

# 2. Convert quantities to binary indicators (1 if bought, 0 if not)
def encode_units(x):
    return 1 if x >= 1 else 0

basket_encoded = basket.applymap(encode_units)
print("Basket matrix dimensions (Orders x Products):", basket_encoded.shape)
basket_encoded.head()

/tmp/ipykernel_1444/3474385926.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket_encoded = basket.applymap(encode_units)


Basket matrix dimensions (Orders x Products): (50000, 20)


ProductID,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
OrderID,,,,,,,,,,,,,,,,,,,,
500001,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
500002,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
500003,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
500004,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
500005,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


# Association Rule Mining via Apriori Algorithm
Extracting frequent itemsets with a minimum support threshold, then computing Confidence and Lift metrics to find strong product associations.

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Lower min_support threshold so rules can be identified
frequent_itemsets = apriori(basket_encoded, min_support=0.001, use_colnames=True)

# 2. Check if itemsets were found
if frequent_itemsets.empty:
    print("No frequent itemsets found. Lower min_support further.")
else:
    # 3. Generate rules with a lower lift threshold
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=0.1)
    rules_sorted = rules.sort_values(by='lift', ascending=False)

    print(f"Total Rules Discovered: {len(rules_sorted)}")
    display(rules_sorted[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

    # 4. Plot rules if any exist
    if not rules_sorted.empty:
        plt.figure(figsize=(9, 5))
        sns.scatterplot(
            data=rules_sorted,
            x='confidence',
            y='lift',
            size='support',
            hue='lift',
            palette='viridis',
            sizes=(40, 200)
        )
        plt.title('Market Basket Rules: Confidence vs. Lift', fontsize=12, fontweight='bold')
        plt.xlabel('Confidence (Probability of buying Item B given Item A)')
        plt.ylabel('Lift (Strength of Association)')
        plt.show()
    else:
        print("No association rules met the threshold to display on the plot.")

Total Rules Discovered: 0


,antecedents,consequents,support,confidence,lift


No association rules met the threshold to display on the plot.


In [9]:
# Check how many distinct products are bought per order
items_per_order = basket_encoded.sum(axis=1)

print("--- Order Basket Size Summary ---")
print(items_per_order.describe())
print("\nOrders with 1 item:", (items_per_order == 1).sum())
print("Orders with 2+ items:", (items_per_order >= 2).sum())

--- Order Basket Size Summary ---
count    50000.000000
mean         0.997900
std          0.045778
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000
dtype: float64

Orders with 1 item: 49895
Orders with 2+ items: 0
